In [1]:
!pip install -q transformers accelerate datasets safetensors einops

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 76.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is

In [2]:
import torch
from datasets import load_dataset
import pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer
)

print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")



2025-11-24 22:47:10.265554: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764024430.628769      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764024430.745656      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

GPU available: True
GPU name: Tesla T4


In [3]:
data_path = "/kaggle/input/dataset/train_english_only.jsonl"   # <-- CHANGE THIS

raw_ds = load_dataset("json", data_files={"train": data_path})
df = raw_ds["train"].to_pandas()

print("Dataset loaded. Total rows:", len(df))
print(df.head())


Generating train split: 0 examples [00:00, ? examples/s]

Dataset loaded. Total rows: 17726
                                         instruction input  \
0  You are a financial sentiment analysis expert....         
1  You are a financial expert. Your task is to pr...         
2  Is it ok to just report to 1 credit bureau ins...         
3  A trust fund is established with a principal o...         
4  Create an extract of a scientific journal arti...         

                                              output  
0  <think>\nOkay, let's see. The user wants me to...  
1  <think>\nOkay, the user is asking again about ...  
2  <think>\nOkay, so the user is asking if it's o...  
3  <think>\nOkay, so I need to figure out how muc...  
4  <think>\nOkay, the user wants me to create an ...  


In [4]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

def build_prompt(row):
    return f"Instruction: {row['instruction']}\nInput: {row['input']}\nOutput: {row['output']}"

def count_tokens(row):
    return len(tokenizer(build_prompt(row)).input_ids)

print("Counting token lengths... (this may take ~3–6 min on T4)")
df["token_len"] = df.apply(count_tokens, axis=1)

print("Token length stats:")
print(df["token_len"].describe())

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Counting token lengths... (this may take ~3–6 min on T4)
Token length stats:
count    17726.000000
mean      2087.469706
std       1838.325355
min        162.000000
25%       1115.000000
50%       1735.000000
75%       2394.000000
max      59864.000000
Name: token_len, dtype: float64


In [5]:
print("Before filtering:", len(df))
df_filtered = df[df["token_len"] <= 2394].copy()
print("After filtering:", len(df_filtered))
print(f"Removed {len(df) - len(df_filtered)} long samples.")

Before filtering: 17726
After filtering: 13296
Removed 4430 long samples.


In [6]:
filtered_path = "/kaggle/working/filtered_dataset.jsonl"
df_filtered.to_json(filtered_path, orient="records", lines=True)

print("Filtered dataset saved to:", filtered_path)

dataset = load_dataset("json", data_files={"train": filtered_path}, split="train")

Filtered dataset saved to: /kaggle/working/filtered_dataset.jsonl


Generating train split: 0 examples [00:00, ? examples/s]

In [7]:
def format_example(example):
    inst = example["instruction"]
    inp = example["input"]
    out = example["output"]
    text = f"Instruction: {inst}\nInput: {inp}\nOutput: {out}"
    return {"text": text}

dataset = dataset.map(format_example)
print("Example formatted text:\n", dataset[0]["text"][:300], "...")

Map:   0%|          | 0/13296 [00:00<?, ? examples/s]

Example formatted text:
 Instruction: You are a financial sentiment analysis expert. Your task is to analyze the sentiment expressed in the given financial text.Only reply with positive, neutral, or negative.Stora Enso Oyj , the largest papermaker , in October said it would close four mills .
Input: 
Output: <think>
Okay, l ...


In [8]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=False,
        padding="max_length",
        max_length=2394,            # Best for Kaggle T4
    )

tokenized = dataset.map(tokenize, batched=True, remove_columns=dataset.column_names)
tokenized = tokenized.map(lambda x: {"labels": x["input_ids"]})

print("Tokenized example keys:", tokenized[0].keys())

Map:   0%|          | 0/13296 [00:00<?, ? examples/s]

Map:   0%|          | 0/13296 [00:00<?, ? examples/s]

Tokenized example keys: dict_keys(['input_ids', 'attention_mask', 'labels'])


In [9]:
# --- Memory-safe training patch ------------------------------------------------
import os, gc, torch

# reduce fragmentation (helps CUDA allocator)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# free memory
torch.cuda.empty_cache()
gc.collect()

# reload or reconfigure model: make sure model is loaded in default precision
# If you already loaded the model with torch_dtype=float16, delete it and reload:
try:
    del model
    torch.cuda.empty_cache()
    gc.collect()
except Exception:
    pass

# load model (let Trainer manage AMP)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    trust_remote_code=True,
)  # <-- no torch_dtype=float16 here

# Important: disable caching and enable gradient checkpointing
if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()
# disable use_cache so model doesn't store past during training
try:
    model.config.use_cache = False
except Exception:
    pass

# free some memory again
torch.cuda.empty_cache()
gc.collect()

# Adjust training args for memory safety
training_args = TrainingArguments(
    output_dir="/kaggle/working/qwen-output",
    num_train_epochs=3,
    per_device_train_batch_size=1,    # reduce to 1
    gradient_accumulation_steps=16,   # effective batch size 16 (1*16)
    learning_rate=2e-5,
    warmup_ratio=0.03,
    logging_steps=50,
    save_steps=800,
    save_total_limit=2,
    fp16=True,                        # let Trainer handle Mixed Precision
    optim="adamw_torch",
    report_to="none",
)

# recreate Trainer with updated args if needed
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
)

# sanity: print GPU memory summary
if torch.cuda.is_available():
    print("Allocated:", torch.cuda.memory_allocated() / (1024**3), "GB")
    print("Reserved: ", torch.cuda.memory_reserved() / (1024**3), "GB")
# --------------------------------------------------------------------------------


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Allocated: 0.8960118293762207 GB
Reserved:  1.41015625 GB


In [10]:
trainer.train()


# ============================================================
# 11. SAVE FINAL MODEL
# ============================================================

save_dir = "/kaggle/working/qwen-final-model"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

print("Training complete! Model saved to:", save_dir)

Step,Training Loss
50,2.464800
100,0.851600
150,0.834500
200,0.792100
250,0.771200
300,0.781900
350,0.770000
400,0.764700
450,0.752000
500,0.766700


Training complete! Model saved to: /kaggle/working/qwen-final-model
